In [2]:
import pandas as pd

# File paths
beta_file = "/scratch/c.c2029098/dementia_ml_project/data/raw/Kunkle_etal_2019_IGAP_Summary_statistics.with_allelefreqs.txt"
allele_file = "/scratch/c.c2029098/dementia_ml_project/data/processed/snps_list/77_snps_effect_allele.txt"
output_file = "/scratch/c.c2029098/dementia_ml_project/data/processed/snps_list/prs_77.txt"

# Load data
beta_df = pd.read_csv(beta_file, delim_whitespace=True, header=0)
allele_df = pd.read_csv(allele_file, sep="\t", header=0)

# Merge on SNP ID
merged = beta_df.merge(
    allele_df[["SNP", "POS", "effect_allele", "non_effect_allele"]],
    left_on="MarkerName",
    right_on="SNP",
    how="inner"
)

# Flip beta if allele directions are swapped (relative to allele file)
def adjust_beta(row):
    ea_beta = str(row["Effect_allele"]).upper()
    nea_beta = str(row["Non_Effect_allele"]).upper()
    ea_ref  = str(row["effect_allele"]).upper()       # from allele file
    nea_ref = str(row["non_effect_allele"]).upper()   # from allele file
    b = row["Beta"]
    if pd.isna(b):
        return None
    if ea_beta == ea_ref and nea_beta == nea_ref:
        return b
    if ea_beta == nea_ref and nea_beta == ea_ref:
        return -b
    return None  # mismatch/ambiguous → drop

merged["Beta_adjusted"] = merged.apply(adjust_beta, axis=1)

# Final PLINK weights: use effect_allele FROM THE ALLELE FILE + pos, sorted by pos
result = (
    merged
    .dropna(subset=["Beta_adjusted"])
    .loc[:, ["SNP", "POS", "effect_allele", "Beta_adjusted"]]
    .rename(columns={"effect_allele": "Effect_allele"})
)

# Save as 4-column weights file
result.to_csv(output_file, sep="\t", index=False)
print(f"Adjusted PRS weights saved to: {output_file}")



/tmp/ipykernel_508/2522179044.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  beta_df = pd.read_csv(beta_file, delim_whitespace=True, header=0)


Adjusted PRS weights saved to: /scratch/c.c2029098/dementia_ml_project/data/processed/snps_list/prs_77.txt
